In [51]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind
cwd = os.getcwd()
data_dir = os.path.join(cwd, 'data')
out_dir = os.path.join(cwd, 'output')

In [52]:
def get_earliest_row(df, date_col):
    # Ensure the column is datetime
    df[date_col] = pd.to_datetime(df[date_col])
    
    # Get index of earliest date
    idx = df[date_col].idxmin()
    
    # Return that row
    return df.loc[idx]
def delta_months(date1, date2):
    d1 = pd.to_datetime(date1)
    d2 = pd.to_datetime(date2)
    
    return (d2.year - d1.year) * 12 + (d2.month - d1.month)

In [53]:
# Load data
raw_data_df = pd.read_csv(os.path.join(data_dir, 'MDS-UPDRS_Part_III_24Jan2024.csv'))
print(f"Total entries in raw data: {len(raw_data_df)}")
print(f"Total unique patients in raw data: {raw_data_df['PATNO'].nunique()}")

# MEDSTATE
raw_data_df['MEDSTATE'] = None
raw_data_df.loc[raw_data_df['PDSTATE'] == 'ON', 'MEDSTATE'] = 'ON'
raw_data_df.loc[
    (raw_data_df['PDSTATE'] == 'OFF') |
    (raw_data_df['PDSTATE'].isna() & (raw_data_df['PDTRTMNT'] == 0)),
    'MEDSTATE'
] = 'OFF'

# Keep relevant columns
columns_of_interest = ['PATNO', 'EVENT_ID', 'INFODT', 'MEDSTATE', 'NP3TOT', 'NHY'] + raw_data_df.columns.tolist()[23:-7]
df = raw_data_df[columns_of_interest].dropna().copy()

# Subscores
df['RIGIDITY'] = df[['NP3RIGN', 'NP3RIGRU', 'NP3RIGLU', 'NP3RIGRL', 'NP3RIGLL']].sum(axis=1)

df['BRADY'] = df[
    ['NP3FTAPR', 'NP3FTAPL', 'NP3HMOVR', 'NP3HMOVL', 'NP3PRSPR',
     'NP3PRSPL', 'NP3TTAPR', 'NP3TTAPL', 'NP3LGAGR', 'NP3LGAGL', 'NP3BRADY']
].sum(axis=1)

df['PIGD'] = df[['NP3RISNG', 'NP3GAIT', 'NP3FRZGT', 'NP3PSTBL', 'NP3POSTR']].sum(axis=1)

df['TREMOR'] = df[
    ['NP3PTRMR', 'NP3PTRML', 'NP3KTRMR', 'NP3KTRML', 'NP3RTARU',
     'NP3RTALU', 'NP3RTARL', 'NP3RTALL', 'NP3RTALJ', 'NP3RTCON']
].sum(axis=1)

# Visit month
visit_dict = {
    'BL': 0, 'V01': 3, 'V02': 6, 'V03': 9, 'V04': 12,
    'V05': 18, 'V06': 24, 'V07': 30, 'V08': 36, 'V09': 42,
    'V10': 48, 'V11': 54, 'V12': 60, 'V13': 72, 'V14': 84,
    'V15': 96, 'V16': 108, 'V17': 120, 'V18': 132, 'V19': 144,
    'V20': 156
}
df['MONTH'] = df['EVENT_ID'].map(visit_dict)

# Keeping only MEDSTATE = OFF
df = df[df['MEDSTATE'] == 'OFF'].copy()

# Keep only first 12 months
df = df[df['MONTH'] <= 12].copy()

# Keep only patients with at least 5 visits
df = df[df.groupby('PATNO')['PATNO'].transform('size') >= 3].copy()

# Baseline NHY from rows where MONTH == 0
baseline_nhy = (
    df[df['MONTH'] == 0]
    .drop_duplicates('PATNO')
    .set_index('PATNO')['NHY']
)
df['NHY_BASELINE'] = df['PATNO'].map(baseline_nhy)

# Define stage from baseline NHY
df['STAGE'] = np.where(df['NHY_BASELINE'] >= 2, 'LATE', 'EARLY')

# saving master df
df.to_csv(os.path.join(data_dir, 'master_df.csv'), index=False)

print(f"Total entries: {len(df)}")
print(f"Total unique patients: {df['PATNO'].nunique()}")

df


Total entries in raw data: 25104
Total unique patients in raw data: 3382
Total entries: 4085
Total unique patients: 1146


,PATNO,EVENT_ID,INFODT,MEDSTATE,NP3TOT,NHY,NP3SPCH,NP3FACXP,NP3RIGN,NP3RIGRU,...,NP3RTALL,NP3RTALJ,NP3RTCON,RIGIDITY,BRADY,PIGD,TREMOR,MONTH,NHY_BASELINE,STAGE
9,3001,BL,03/2011,OFF,12.0,1.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,2.0,5.0,1.0,2.0,0.0,1.0,EARLY
11,3001,V01,05/2011,OFF,18.0,2.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,4.0,7.0,1.0,4.0,3.0,1.0,EARLY
12,3001,V02,08/2011,OFF,23.0,2.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,4.0,9.0,1.0,7.0,6.0,1.0,EARLY
13,3001,V03,11/2011,OFF,19.0,2.0,0.0,2.0,0.0,2.0,...,0.0,0.0,0.0,3.0,7.0,1.0,6.0,9.0,1.0,EARLY
14,3001,V04,03/2012,OFF,20.0,2.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,3.0,8.0,1.0,6.0,12.0,1.0,EARLY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24106,181822,V02,07/2023,OFF,24.0,2.0,0.0,3.0,1.0,0.0,...,0.0,0.0,2.0,2.0,12.0,2.0,5.0,6.0,2.0,LATE
24108,181822,V04,01/2024,OFF,21.0,2.0,0.0,2.0,1.0,0.0,...,0.0,0.0,3.0,4.0,8.0,1.0,6.0,12.0,2.0,LATE
24125,182341,BL,01/2023,OFF,8.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,4.0,0.0,1.0,0.0,1.0,EARLY
24127,182341,V02,07/2023,OFF,13.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,6.0,2.0,2.0,6.0,1.0,EARLY


In [54]:
# Plotting for visualizing results
%matplotlib qt

# Checking sample size for each stage
early_df = df[df['STAGE'] == 'EARLY']
late_df = df[df['STAGE'] == 'LATE']

print(f"Early stage entries: {len(early_df)} \nEarly stage unique patients: {early_df['PATNO'].nunique()}")
print(f"Late stage entries: {len(late_df)} \nLate stage unique patients: {late_df['PATNO'].nunique()}")


Early stage entries: 2545 
Early stage unique patients: 720
Late stage entries: 1540 
Late stage unique patients: 426


## Baseline charactristics

In [55]:
def summarize_baseline_groups(early_df, late_df, measures):
    """
    Create summary table with mean ± std for:
    - All cohort
    - Early group
    - Late group
    
    And perform independent t-tests (Early vs Late).

    Parameters
    ----------
    early_df : pd.DataFrame
    late_df : pd.DataFrame
    measures : list of str

    Returns
    -------
    summary_df : pd.DataFrame
    """

    results = []

    # Combine for "All cohort"
    all_df = pd.concat([early_df, late_df], axis=0)

    for m in measures:
        # Drop NaNs
        early_vals = early_df[m].dropna()
        late_vals = late_df[m].dropna()
        all_vals = all_df[m].dropna()

        # Means and STDs
        early_mean = early_vals.mean()
        early_std = early_vals.std()

        late_mean = late_vals.mean()
        late_std = late_vals.std()

        all_mean = all_vals.mean()
        all_std = all_vals.std()

        # T-test (Welch by default -> unequal variance safer)
        if len(early_vals) > 1 and len(late_vals) > 1:
            t_stat, p_val = ttest_ind(early_vals, late_vals, equal_var=False)
        else:
            t_stat, p_val = np.nan, np.nan

        results.append({
            "Measure": m,
            "All_mean": all_mean,
            "All_std": all_std,
            "Early_mean": early_mean,
            "Early_std": early_std,
            "Late_mean": late_mean,
            "Late_std": late_std,
            "t_stat": t_stat,
            "p_value": p_val
        })

    summary_df = pd.DataFrame(results)

    return summary_df

In [56]:
summary_df = summarize_baseline_groups(
    early_df=early_df[early_df['MONTH'] == 0],
    late_df=late_df[late_df['MONTH'] == 0],
    measures=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR']
)
summary_df.to_csv(os.path.join(out_dir, 'baseline_summary_df.csv'), index=False)
summary_df

,Measure,All_mean,All_std,Early_mean,Early_std,Late_mean,Late_std,t_stat,p_value
0,NP3TOT,13.676265,11.805146,7.041667,7.296497,24.889671,9.207646,-34.161836,3.293246e-154
1,RIGIDITY,2.329843,2.650421,1.133333,1.625253,4.352113,2.816415,-21.559930,1.241888e-76
2,BRADY,6.237347,6.187143,2.887500,3.416374,11.899061,5.678579,-29.725425,1.017436e-120
3,PIGD,1.053229,1.346434,0.616667,0.953874,1.791080,1.573610,-13.960836,1.167238e-38
4,TREMOR,3.028796,3.319395,1.852778,2.580119,5.016432,3.481340,-16.294529,7.143798e-51


## Progression Analysis
Subscore ~ MONTH + STAGE + MONTH:STAGE

In [73]:
def plot_progression_subplots(early_df_plot, late_df_plot, measures, y_labels):
    """
    Plot regression lines with CI for multiple measures in one figure.

    Parameters
    ----------
    early_df_plot : pd.DataFrame
    late_df_plot : pd.DataFrame
    measures : list of str
    y_labels : list of str (same length as measures)
    """

    n = len(measures)

    # Auto layout (2 columns looks clean for papers)
    n_cols = 2
    n_rows = int(np.ceil(n / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
    axes = axes.flatten()

    for i, (measure, ylab) in enumerate(zip(measures, y_labels)):
        ax = axes[i]

        # Early
        sns.regplot(
            data=early_df_plot, x='MONTH', y=measure,
            scatter=False,
            ci=95,
            color='b',
            label='Early Stage',
            line_kws={'linewidth': 2},
            ax=ax
        )

        # Mid-stage (rename if you updated terminology)
        sns.regplot(
            data=late_df_plot, x='MONTH', y=measure,
            scatter=False,
            ci=95,
            color='r',
            label='Mid Stage',
            line_kws={'linewidth': 2},
            ax=ax
        )

        ax.set_xlabel('Months Since Baseline')
        ax.set_ylabel(ylab)
        ax.set_title(f'{ylab}')

        ax.legend()
        ax.grid(alpha=0.2)

    # Remove empty subplots if any
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [74]:
# PLotting for visualizing results (regression with 95%CI)
early_participants = early_df['PATNO'].unique().tolist()
early_df_plot = early_df.copy()
for p in early_participants:
    part_df = early_df_plot[early_df_plot['PATNO'] == p]
    part_baseline = part_df[part_df['MONTH'] == 0]
    for measure in ['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR']:
        if measure in part_baseline.columns:
            baseline_value = part_baseline[measure].values[0]
            early_df_plot.loc[early_df_plot['PATNO'] == p, measure] = early_df_plot.loc[early_df_plot['PATNO'] == p, measure] - baseline_value


late_participants = late_df['PATNO'].unique().tolist()
late_df_plot = late_df.copy()
for p in late_participants:
    part_df = late_df_plot[late_df_plot['PATNO'] == p]
    part_baseline = part_df[part_df['MONTH'] == 0]
    for measure in ['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR']:
        if measure in part_baseline.columns:
            baseline_value = part_baseline[measure].values[0]
            late_df_plot.loc[late_df_plot['PATNO'] == p, measure] = late_df_plot.loc[late_df_plot['PATNO'] == p, measure] - baseline_value

measures = ['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR']
y_labels = ['Total MDS-UPDRS Part III Change', 'Rigidity Subscore Change', 'Bradykinesia Subscore Change', 'PIGD Subscore Change', 'Tremor Subscore Change']
plot_progression_subplots(
    early_df_plot,
    late_df_plot,
    measures,
    y_labels
)

In [58]:
def multiple_dvs_lmm(df, dvs, ivs = ' ~ MONTHS_SINCE_BASELINE*STAGE', 
                     random_intercept='PATNO', random_slope='~MONTH', std_dv = False):
    """
    Fit linear mixed models (LMMs) for multiple dependent variables (DVs).

    Parameters:
    df (pd.DataFrame): DataFrame containing the data for modeling.
    dvs (list of str): List of dependent variable column names to model.
    ivs (str): Fixed effects formula (default includes Occasion, Age, and Sex).
    random_intercept (str): Column name to use for random intercept grouping (e.g., participant ID).
    random_slope (str): Random slope formula (default is '~Occasion').

    Returns:
    list: List of fitted LMM model results (MixedLMResults objects).
    """
    mdfs = []
    scaler = StandardScaler()

    for dv in dvs:

        # Standardize the dependent variable if specified
        if std_dv:
            standardized_col = dv + "_z"
            df[standardized_col] = scaler.fit_transform(df[[dv]])
            formula = standardized_col + ivs
        else:
            formula = dv + ivs # Construct the full formula: e.g., 'Measure ~ Occasion + Age + Sex'
        
        md = mixedlm(formula, data=df, groups=df[random_intercept], re_formula=random_slope)
        mdf = md.fit(method='lbfgs')
        mdfs.append(mdf)

    return mdfs

def univariate_lmm_dataframe(dvs, mdfs, output=None, save=False):
    """
    Create a summary DataFrame of parameter estimates and p-values for each DV's LMM.

    Parameters:
    dvs (list of str): List of dependent variable names corresponding to the models.
    mdfs (list): List of fitted LMM results.
    output (str): File path to save the output Excel file (if save=True).
    save (bool): Whether to save the summary DataFrame to an Excel file.

    Returns:
    pd.DataFrame: Multi-index DataFrame summarizing parameters and p-values per DV.
    """
    columns  = []
    ivs = mdfs[0].pvalues.dropna().index # Assume all models have the same IVs
    for iv in ivs:
        columns.append((iv, 'param')) # Parameter estimate
        columns.append((iv,'pval')) # p-value
        columns.append((iv, 'pval_fdr')) # FDR-corrected p-value
    
    df = pd.DataFrame(index=dvs, columns=pd.MultiIndex.from_tuples(columns))
    for dv, mdf in zip(dvs, mdfs):
        for iv in ivs:
            df.loc[dv, (iv, 'param')] = round(mdf.params[iv],3)
            df.loc[dv, (iv, 'pval')] = round(mdf.pvalues[iv],3)

        # Apply BH-FDR correction per IV across DVs
    for iv in ivs:
        # Convert to numeric, force None -> NaN
        pvals = pd.to_numeric(df[(iv, 'pval')], errors='coerce')

        # Mask valid p-values
        valid_mask = pvals.notna()

        # Initialize corrected p-values with NaN
        pvals_fdr = np.full(len(pvals), np.nan)

        # Apply FDR only to valid p-values
        if valid_mask.sum() > 0:
            _, pvals_fdr_valid, _, _ = multipletests(
                pvals[valid_mask].values,
                method='fdr_bh'
            )
            pvals_fdr[valid_mask] = pvals_fdr_valid

        # Assign back to DataFrame
        df[(iv, 'pval_fdr')] = pvals_fdr

        # Round raw p-values last
        df[(iv, 'pval')] = pvals
    
    if save:
        df.to_excel(output)
    return df

In [59]:
mdfs = multiple_dvs_lmm(df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], ivs=' ~ MONTH*STAGE')
summary_df = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], mdfs=mdfs, output=os.path.join(out_dir, 'lmm_summary.xlsx'), save=True)
summary_df

c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Intercept               STAGE[T.LATE]                MONTH       \
             param pval pval_fdr         param pval pval_fdr  param pval   
NP3TOT        7.21  0.0      0.0        17.556  0.0      0.0   0.15  0.0   
RIGIDITY      1.19  0.0      0.0          3.18  0.0      0.0  0.026  0.0   
BRADY        2.972  0.0      0.0         8.843  0.0      0.0  0.066  0.0   
PIGD         0.656  0.0      0.0         1.104  0.0      0.0  0.017  0.0   
TREMOR       1.851  0.0      0.0         3.129  0.0      0.0  0.032  0.0   

                  MONTH:STAGE[T.LATE]  ...          Group Var                \
         pval_fdr               param  ... pval_fdr     param pval pval_fdr   
NP3TOT        0.0               0.093  ...  0.02250     3.149  0.0      0.0   
RIGIDITY      0.0               0.019  ...  0.11000     2.018  0.0      0.0   
BRADY         0.0               0.033  ...  0.13125      2.55  0.0      0.0   
PIGD          0.0               0.022  ...  0.00500     1.619  0.0      0.0   
TREMOR        0.0               0.002  ...  0.87300     3.127  0.0      0.0   

         Group x MONTH Cov                 MONTH Var                
                     param   pval pval_fdr     param pval pval_fdr  
NP3TOT               0.056  0.000    0.000     0.006  0.0      0.0  
RIGIDITY              0.02  0.004    0.005     0.004  0.0      0.0  
BRADY                0.013  0.105    0.105     0.005  0.0      0.0  
PIGD                 0.019  0.004    0.005     0.006  0.0      0.0  
TREMOR               0.045  0.000    0.000     0.005  0.0      0.0  

[5 rows x 21 columns]

In [60]:
# Full cohort
mdfs = multiple_dvs_lmm(df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_all = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs, output=os.path.join(out_dir, 'lmm_all.xlsx'), 
                                      save=True)

results_all

c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Intercept                MONTH                Group Var              
             param pval pval_fdr  param  pval pval_fdr     param pval pval_fdr
NP3TOT      13.703  0.0      0.0  0.197  0.25     0.25    14.124  0.0      0.0
RIGIDITY     2.373  0.0      0.0  0.033  0.00     0.00     3.416  0.0      0.0
BRADY        6.265  0.0      0.0  0.078  0.00     0.00     3.602  0.0      0.0
PIGD         1.068  0.0      0.0  0.024  0.00     0.00     2.085  0.0      0.0
TREMOR       3.017  0.0      0.0  0.032  0.00     0.00     4.145  0.0      0.0

In [50]:
# df = raw_data_df[columns_of_interest].dropna().copy()

# # Subscores
# df['RIGIDITY'] = df[['NP3RIGN', 'NP3RIGRU', 'NP3RIGLU', 'NP3RIGRL', 'NP3RIGLL']].sum(axis=1)

# df['BRADY'] = df[
#     ['NP3FTAPR', 'NP3FTAPL', 'NP3HMOVR', 'NP3HMOVL', 'NP3PRSPR',
#      'NP3PRSPL', 'NP3TTAPR', 'NP3TTAPL', 'NP3LGAGR', 'NP3LGAGL', 'NP3BRADY']
# ].sum(axis=1)

# df['PIGD'] = df[['NP3RISNG', 'NP3GAIT', 'NP3FRZGT', 'NP3PSTBL', 'NP3POSTR']].sum(axis=1)

# df['TREMOR'] = df[
#     ['NP3PTRMR', 'NP3PTRML', 'NP3KTRMR', 'NP3KTRML', 'NP3RTARU',
#      'NP3RTALU', 'NP3RTARL', 'NP3RTALL', 'NP3RTALJ', 'NP3RTCON']
# ].sum(axis=1)

# # Visit month
# visit_dict = {
#     'BL': 0, 'V01': 3, 'V02': 6, 'V03': 9, 'V04': 12,
#     'V05': 18, 'V06': 24, 'V07': 30, 'V08': 36, 'V09': 42,
#     'V10': 48, 'V11': 54, 'V12': 60, 'V13': 72, 'V14': 84,
#     'V15': 96, 'V16': 108, 'V17': 120, 'V18': 132, 'V19': 144,
#     'V20': 156
# }
# df['MONTH'] = df['EVENT_ID'].map(visit_dict)

# # Keeping only MEDSTATE = OFF
# df = df[df['MEDSTATE'] == 'OFF'].copy()

# # Keep only first 12 months
# df = df[df['MONTH'] < 12].copy()

# # Keep only patients with at least 5 visits
# df = df[df.groupby('PATNO')['PATNO'].transform('size') >= 3].copy()

# mdfs = multiple_dvs_lmm(df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
#                         ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
# results_all = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
#                                       mdfs=mdfs, output=os.path.join(out_dir, 'lmm_all.xlsx'), 
#                                       save=True)

# results_all